# 推論段階の防御（Inference Stage Defense）

推論段階では入力サンプルのみを制御でき、悪意のある入力（敵対的サンプルまたはバックドアトリガー入力）を拒否・修正する防御を行う。

## バックドアに対する推論時防御

### 毒入れサンプルの検出（Poisoned Sample Detection）

推論時に、入力がバックドアトリガーを含むかどうかを判定する。

#### クエリ摂動ベース

入力に摂動を加え、予測の安定性の違いで検出する。

| 手法 | アプローチ |
|------|----------|
| **STRIP** | 入力に様々な画像パターンを重ね合わせ、予測のエントロピーを計算。クリーンな入力は高エントロピー、バックドアトリガー入力は低エントロピー |
| **SentiNet** | GradCAMで顕著領域を局所化して検出 |
| **TeCo** | Corruption robustness consistencyを測定 |
| **SCALE-UP** | ピクセル単位の増幅プロセスで検出 |

#### クエリ区別ベース

入力そのものの特性からトリガーの有無を判定する。

| 手法 | アプローチ |
|------|----------|
| **FreqDetector** | 高周波アーティファクトを検出 |
| **FREAK** | 周波数感度の差異を分析 |
| **Orion** | 多分岐ネットワークに小型ネットワークを装着して検出 |

### 毒入れサンプルの復旧（Poisoned Sample Recovery）

バックドア入力を検出した場合に、正しい予測を復元する。

| 手法 | アプローチ |
|------|----------|
| **Orion（再ラベリング）** | 浅層の特徴から元のラベルを推論 |
| **NAB** | 2つの再ラベリング手法を提案 |
| **ZIP（浄化）** | 線形変換でトリガーパターンを破壊し、事前学習済み拡散モデルで意味情報を補完 |

## 敵対的サンプルに対する推論時防御

### 正しい予測のフィードバック（Correct Feedback）

敵対的な摂動を除去または無力化して正しい予測を返す。3つのアプローチがある。

#### 入力変換（Input Transformation）

入力に変換 $T(\cdot)$ を適用して摂動を除去する。

$$
\hat{y} = f(T(\mathbf{x}_{\text{test}}))
$$

| 手法 | 変換 |
|------|------|
| **Guo et al.** | 画像のクロッピング、リスケーリング、ビット深度削減、JPEG圧縮 |
| **Feature Distillation** | JPEG圧縮ベース |
| **DAD** | 高周波成分の除去 |
| **Song et al.** | Saak変換 |
| **Mustafa et al.** | 超解像技術 |

#### 入力再構成（Input Reconstruction）

生成モデル $R(\cdot)$ でクリーンな入力を再構成する。

$$
\hat{y} = f(R(\mathbf{x}_{\text{test}})), \quad R^* = \arg\min_R \mathbb{E}_{\mathbf{x} \sim p_{\text{data}}} \|R(\mathbf{x}) - \mathbf{x}\|^2
$$

| 手法 | アプローチ |
|------|----------|
| **MagNet** | オートエンコーダベースのreformer |
| **Defense-GAN** | GANによる分布近似 |
| **ComDefend** | 圧縮ベースの分布近似 |
| **Hill et al.** | Energy-Based Model（EBM）で長期Langevin更新 |
| **DiffPure** | 拡散モデルによるデノイジング。明示的な分布近似 |
| **HGD** | 高レベル表現で誘導するデノイザー |
| **APE-GAN** | GANベースのデノイザー |
| **DISCO** | 局所陰関数でRGB値を予測 |
| **D3** | 辞書学習と疎再構成 |

#### 入力摂動（Input Perturbation）

摂動を打ち消す逆方向の摂動 $\gamma$ を最適化する。

$$
\gamma^* = \arg\min_\gamma \mathcal{L}_{\text{task}}(\mathbf{x}_{\text{test}} + \gamma)
$$

| 手法 | アプローチ |
|------|----------|
| **PixelDefend** | PixelCNNで分布を推定し、クリーンな分布に近い入力に摂動 |
| **HPD** | Hilbertベースの改良版PixelCNN |
| **SOAP** | 自己教師あり学習タスクの最適化 |
| **Hedge Defense** | 決定境界のシフト |

### 入力の拒否（Rejecting Input / Detection）

敵対的サンプルを検出して拒否する。

#### 特徴ベースの検出

| 手法 | アプローチ |
|------|----------|
| **Metzen et al.** | 補助的なバイナリ分類器を訓練 |
| **SafetyNet** | 離散的な特徴量子化 |
| **Grosse et al.** | 追加クラスの導入 |
| **Li et al.** | PCAによる統計量抽出 |
| **LID** | 局所的な内在次元（Local Intrinsic Dimensionality）の違いを利用 |
| **Zhao et al.** | フィッシャー情報行列の固有値 |
| **ML-LOO** | 特徴帰属の分散 |
| **LNG** | Latent Neighborhood Graphの構築 |
| **Harder et al.** | フーリエ係数の解析 |

#### 分布の差異に基づく検出

| 手法 | アプローチ |
|------|----------|
| **Grosse et al.** | 最大平均偏差（MMD）とenergy distance |
| **Gao et al.** | 効果的なカーネルを活用 |
| **Feinman et al.** | カーネル密度推定 |

### 悪意のあるクエリ列への防御

モデル窃取攻撃のように、一連のクエリを通じてモデルの情報を抽出しようとする攻撃への防御。

- 異常なクエリパターンの検出
- レート制限
- 出力の摂動（信頼度スコアの丸め等）
- ウォーターマーキング

## 参考文献

- [Wu et al. (2023). Defenses in Adversarial Machine Learning: A Survey. Section VI.](https://arxiv.org/abs/2312.08890)
- [Gao et al. (2019). STRIP: A Defence Against Trojan Attacks on Deep Neural Networks.](https://arxiv.org/abs/1902.06531)
- [Nie et al. (2022). DiffPure: Diffusion Models Beat Adversarial Purification.](https://arxiv.org/abs/2205.07460)
- [Meng & Chen (2017). MagNet: a Two-Pronged Defense against Adversarial Examples.](https://arxiv.org/abs/1705.09064)
- [Ma et al. (2018). Characterizing Adversarial Subspaces Using Local Intrinsic Dimensionality.](https://arxiv.org/abs/1801.02613)